<a href="https://colab.research.google.com/github/vc-alejandro/churn_analysis/blob/main/telco_churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prediccion de Churn - Telco Customer Churn
## M2 - Parte I: Implementacion sin framework (Arbol de Decision)

Este notebook contiene el analisis exploratorio, entrenamiento, evaluacion
y comparacion del arbol de decision implementado manualmente en
`decision_tree.py` (sin usar librerias de aprendizaje automatico).

El algoritmo en si (`decision_tree.py`) esta implementado por separado y
corre solo con un compilador de Python, sin depender de este notebook.

## 0. Conexion con GitHub

Esta celda descarga `decision_tree.py` y el dataset directamente desde el
repositorio de GitHub si no se encuentran en el entorno actual (por
ejemplo, al abrir este notebook desde el boton "Open in Colab" de arriba,
donde el repo no esta clonado por defecto).

In [ ]:
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/vc-alejandro/churn_analysis/main"

archivos_necesarios = ["decision_tree.py", "WA_Fn-UseC_-Telco-Customer-Churn.csv"]

for nombre_archivo in archivos_necesarios:
    if not os.path.exists(nombre_archivo):
        url = f"{REPO_RAW}/{nombre_archivo}"
        urllib.request.urlretrieve(url, nombre_archivo)
        print(f"Descargado {nombre_archivo} desde GitHub")
    else:
        print(f"{nombre_archivo} ya disponible localmente")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from decision_tree import DecisionTree

pd.set_option('display.max_columns', None)

## 1. Carga de datos

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

## 2. Limpieza

`TotalCharges` viene como texto porque 11 filas tienen un espacio en
blanco en vez de un numero. Esas 11 filas corresponden a clientes con
`tenure = 0` (clientes nuevos, aun no se les ha cobrado), asi que se
imputan a 0 en vez de eliminarse.

In [ ]:
blancos = df['TotalCharges'].str.strip() == ''
print('Filas con TotalCharges en blanco:', blancos.sum())
df.loc[blancos, ['customerID', 'tenure', 'TotalCharges']]

In [ ]:
df['TotalCharges'] = df['TotalCharges'].str.strip().replace('', '0')
df['TotalCharges'] = df['TotalCharges'].astype(float)

# customerID no es una feature, es solo un identificador
df = df.drop(columns=['customerID'])

## 3. Analisis exploratorio (EDA)

In [ ]:
conteo_churn = df['Churn'].value_counts()
print(conteo_churn)
print('\nProporcion:')
print(df['Churn'].value_counts(normalize=True).round(3))

conteo_churn.plot(kind='bar', color=['#4C72B0', '#DD8452'])
plt.title('Distribucion de la variable objetivo (Churn)')
plt.ylabel('Numero de clientes')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Churn por tipo de contrato
tabla = pd.crosstab(df['Contract'], df['Churn'], normalize='index')
tabla.plot(kind='bar', stacked=True, color=['#4C72B0', '#DD8452'])
plt.title('Proporcion de Churn por tipo de contrato')
plt.ylabel('Proporcion')
plt.xticks(rotation=0)
plt.legend(title='Churn')
plt.show()

In [ ]:
# Distribucion de tenure (antiguedad) separada por Churn
fig, ax = plt.subplots()
df[df['Churn']=='No']['tenure'].hist(alpha=0.6, bins=30, label='No', ax=ax)
df[df['Churn']=='Yes']['tenure'].hist(alpha=0.6, bins=30, label='Yes', ax=ax)
ax.set_xlabel('Tenure (meses)')
ax.set_ylabel('Numero de clientes')
ax.set_title('Antiguedad del cliente vs Churn')
ax.legend(title='Churn')
plt.show()

## 4. Separacion train / test

80% entrenamiento / 20% prueba, estratificado por la variable `Churn`
para conservar la misma proporcion de clases en ambos conjuntos.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# reset_index para que el arbol manual pueda indexar filas 0..n-1
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Entrenamiento del arbol manual (`decision_tree.py`)

`max_depth` y `min_samples_split` se eligieron para evitar overfitting
dado el tamano del dataset (~5,600 ejemplos de entrenamiento).

In [ ]:
arbol = DecisionTree(max_depth=5, min_samples_split=20)
arbol.fit(X_train, y_train)
print('Arbol entrenado.')

In [ ]:
arbol.imprimir_arbol()

## 6. Predicciones y matriz de confusion (arbol manual)

In [ ]:
predicciones = arbol.predict(X_test)
y_test_list = y_test.tolist()

In [ ]:
def matriz_confusion(y_real, y_pred, clase_positiva='Yes'):
    """
    Matriz de confusion 2x2 calculada manualmente.
    clase_positiva define cual etiqueta se considera "positiva"
    (aqui, un cliente que si hace churn).
    """
    vp = sum(1 for r, p in zip(y_real, y_pred) if r == clase_positiva and p == clase_positiva)
    vn = sum(1 for r, p in zip(y_real, y_pred) if r != clase_positiva and p != clase_positiva)
    fp = sum(1 for r, p in zip(y_real, y_pred) if r != clase_positiva and p == clase_positiva)
    fn = sum(1 for r, p in zip(y_real, y_pred) if r == clase_positiva and p != clase_positiva)
    return vp, vn, fp, fn


def metricas(vp, vn, fp, fn):
    accuracy = (vp + vn) / (vp + vn + fp + fn)
    precision = vp / (vp + fp) if (vp + fp) > 0 else 0.0
    recall = vp / (vp + fn) if (vp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

In [ ]:
vp, vn, fp, fn = matriz_confusion(y_test_list, predicciones)
print(f'Verdaderos Positivos (Churn=Yes correcto): {vp}')
print(f'Verdaderos Negativos (Churn=No correcto):  {vn}')
print(f'Falsos Positivos (predijo Yes, era No):    {fp}')
print(f'Falsos Negativos (predijo No, era Yes):    {fn}')

res_manual = metricas(vp, vn, fp, fn)
for k, v in res_manual.items():
    print(f'{k}: {v:.3f}')

In [ ]:
# Visualizacion de la matriz de confusion
matriz = np.array([[vn, fp], [fn, vp]])
fig, ax = plt.subplots()
im = ax.imshow(matriz, cmap='Blues')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred: No', 'Pred: Yes'])
ax.set_yticklabels(['Real: No', 'Real: Yes'])
ax.set_title('Matriz de confusion - Arbol manual')

for i in range(2):
    for j in range(2):
        ax.text(j, i, matriz[i, j], ha='center', va='center', color='black', fontsize=14)

plt.colorbar(im)
plt.show()

## 7. Comparacion contra scikit-learn

Mismo criterio (`entropy`), misma profundidad maxima y mismo
`min_samples_split`, para que la comparacion sea justa. La unica
diferencia real es que sklearn requiere las categoricas codificadas
como numeros (`LabelEncoder`), mientras que el arbol manual las
compara directamente como texto.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()
columnas_categoricas = X_encoded.select_dtypes(include=['object']).columns

for col in columnas_categoricas:
    X_encoded[col] = LabelEncoder().fit_transform(X_encoded[col])

Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

modelo_sklearn = DecisionTreeClassifier(
    criterion='entropy', max_depth=5, min_samples_split=20, random_state=42
)
modelo_sklearn.fit(Xe_train, ye_train)
pred_sklearn = modelo_sklearn.predict(Xe_test)

In [ ]:
vp2, vn2, fp2, fn2 = matriz_confusion(ye_test.tolist(), pred_sklearn.tolist())
res_sklearn = metricas(vp2, vn2, fp2, fn2)

comparacion = pd.DataFrame({
    'Arbol manual': res_manual,
    'sklearn (entropy)': res_sklearn,
})
comparacion

## 8. Analisis y conclusion

*(Completar con la interpretacion de resultados: que tan bien
predice el modelo, que metrica importa mas para este problema de
negocio (recall vs precision, dado el costo de la tarjeta de $5 USD
mencionado en el caso de negocio de courier churn), y que tan
parecido es el desempeno del arbol manual contra scikit-learn.)*